# Synthetic billing reconciliation: ACL to Python/Jupyter

This public notebook demonstrates the pattern used while migrating selected financial controls from ACL Analytics to Python and Jupyter. All identifiers, amounts, rules, and source names are synthetic. The production implementation and credentials are deliberately excluded.

## 1. Control design

The control compares an expected scope with a billed scope at account level. It validates both record counts and monetary amounts, uses a full outer join to detect differences in both directions, applies explicit justification categories, and leaves unresolved cases for review.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import HTML, display

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root))

from python_jupyter.src.control_reconciliation import (
    build_control_kpis,
    reconcile_scopes,
)

## 2. Read sanitized inputs

The portfolio uses CSV fixtures so the example is reproducible without infrastructure access. In the authorized environment, equivalent DataFrames are extracted from Oracle with parameterized SQL and credentials supplied outside the notebook. See `python_jupyter/oracle_connection.example.py`.

In [ ]:
expected_scope = pd.read_csv(repo_root / 'data' / 'synthetic_expected_scope.csv')
billed_scope = pd.read_csv(repo_root / 'data' / 'synthetic_billed_scope.csv')

display(expected_scope)
display(billed_scope)

## 3. Reconcile in both directions

A full outer join is a control decision, not a convenience: it finds accounts present only in the expected universe and accounts present only in billing. Checking only grand totals can hide offsetting differences.

In [ ]:
control = reconcile_scopes(expected_scope, billed_scope)
control[[
    'account_ref', 'expected_line_count', 'actual_line_count',
    'expected_amount', 'actual_amount', 'line_difference',
    'amount_difference', 'status', 'difference_direction'
]]

## 4. Apply and explain business justifications

The billed input separates ordinary rows from synthetic justification types such as cash payments or manual adjustments. A case is marked `JUSTIFIED` only when those rows close both the line-count and amount differences. The rule remains traceable instead of silently deleting an exception.

In [ ]:
control.loc[
    control['status'].isin(['JUSTIFIED', 'PENDING_REVIEW']),
    [
        'account_ref', 'base_amount', 'justified_amount',
        'actual_amount', 'amount_difference', 'status',
        'difference_direction',
    ],
]

## 5. Line, amount, and exception checks

In this fixture the total expected and actual line counts are both 10, but two exceptions still exist: one expected account is missing from billing and one unexpected account was billed. The account-level bidirectional check prevents a false positive that a totals-only control would miss.

In [ ]:
kpis = build_control_kpis(control)
kpis

## 6. Compact Jupyter dashboard

The HTML layer makes the result easier to review while preserving the detailed DataFrame for analysis and export.

In [ ]:
cards = ''.join(
    f"<div class='card'><strong>{label.replace('_', ' ').title()}</strong><br>{value:,.0f}</div>"
    for label, value in kpis.items()
)
pending_html = control.loc[
    control['status'].eq('PENDING_REVIEW'),
    ['account_ref', 'line_difference', 'amount_difference', 'difference_direction'],
].to_html(index=False)
display(HTML(f"""
<style>
.cards {{ display:flex; flex-wrap:wrap; gap:10px; margin:12px 0; }}
.card {{ background:#f4f7fb; border-left:4px solid #2563eb; padding:12px; min-width:150px; }}
</style>
<h3>Synthetic monthly control</h3>
<div class='cards'>{cards}</div>
<h4>Pending review</h4>
{pending_html}
"""))

## 7. Production hardening roadmap

The next engineering steps are modular logging, controlled error handling, parameterized periods, automated tests, execution scheduling, and approved persistence of historical results in Oracle. Historical-table design is being evaluated collaboratively with BI; this repository does not claim that it is already deployed.